# HRM-Text-1B in KerasHub

This notebook converts the pinned Apache-2.0 `sapientinc/HRM-Text-1B` checkpoint to KerasHub in memory, verifies Keras logits against Transformers, and runs greedy cached generation. Use a GPU runtime (a T4 is sufficient for the short sequence used here).

The official checkpoint is a pre-alignment PrefixLM model, not a chat model. The generation cell marks the prompt as a bidirectional PrefixLM prefix, which matches pretraining-time prefill semantics. HRM-Text inherits a Qwen-style vocabulary, but these examples use only the active HRM inference subset documented below.

## Tokenizer vocabulary and inference contract

These examples explicitly format prompts as `<|im_start|><condition>instruction<|im_end|>`, then let the model generate a response that may end with `<|box_end|>`.

| Token | Active inference role |
| --- | --- |
| `<|im_start|>` | Explicit start of the PrefixLM prompt envelope. |
| `<|im_end|>` | Explicit end of the PrefixLM prompt/instruction block. |
| `<|box_end|>` | Checkpoint response terminator and generation stop token. |
| `<|endoftext|>` | KerasHub padding token only; it is not prompt or response content. |
| `<|object_ref_start|>` | `direct` condition. |
| `<|object_ref_end|>` | `cot` condition. |
| `<|quad_start|>` | `noisy` condition. |
| `<|quad_end|>` | `synth` condition. |

The inherited but unsupported tokens in these examples are legacy `<|PAD|>` and `<|direct|>`, `<|cot|>`, `<|noisy|>`, `<|synth|>` labels; reserved `<|box_start|>`; vision `<|vision_start|>`, `<|vision_end|>`, `<|vision_pad|>`, `<|image_pad|>`, `<|video_pad|>`; fill-in-the-middle `<|fim_prefix|>`, `<|fim_middle|>`, `<|fim_suffix|>`, `<|fim_pad|>`; repository `<|repo_name|>`, `<|file_sep|>`; tool `<tool_call>`, `</tool_call>`, `<tool_response>`, `</tool_response>`; and reasoning `<think>`, `</think>`. `<tool_response>` is only checkpoint vocabulary here: no tool-use protocol is enabled, and `<tool_result>` is not a checkpoint special token.

Responsibility boundary: the notebook selects the condition and writes the complete envelope; `HrmTextTokenizer` atomically encodes its registered special tokens; the model consumes token IDs and `token_type_ids` for PrefixLM attention. Neither KerasHub tokenizer nor model injects prompt tokens. The Hugging Face tokenizer is retained for checkpoint conversion and parity/reference checks.

In [ ]:
import json
import os
import subprocess

# Pin an immutable implementation revision for a reproducible run.
REPOSITORY = "https://github.com/pzarzycki/keras-hub.git"
KERAS_HUB_REVISION = "3fd46ba9e4360fbfe808d99f3c97d15aba4ae285"
WORKDIR = "/content/keras-hub"

subprocess.run(["bash", "-lc", "command -v uv >/dev/null || curl -LsSf https://astral.sh/uv/install.sh | sh"], check=True)
os.environ["PATH"] = f"{os.path.expanduser('~/.local/bin')}:{os.environ['PATH']}"
subprocess.run(["git", "clone", REPOSITORY, WORKDIR], check=True)
subprocess.run(["git", "-C", WORKDIR, "checkout", "--detach", KERAS_HUB_REVISION], check=True)
# Pin Transformers without globally upgrading Colab's compiled scientific stack.
subprocess.run(["uv", "pip", "install", "--system", "-q", "transformers==5.9.0", "safetensors"], check=True)
subprocess.run(["uv", "pip", "install", "--system", "-q", "-e", WORKDIR], check=True)

# Keras must select its backend before it is imported.
os.environ["KERAS_BACKEND"] = "torch"


In [ ]:
import gc
import os
import time
os.chdir(WORKDIR)

import numpy as np
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer

if not torch.cuda.is_available():
    raise RuntimeError("Select a Colab GPU runtime before continuing.")

torch.set_default_device("cuda")
import keras
from keras_hub.src.models.hrm_text.hrm_text_causal_lm import HrmTextCausalLM
from keras_hub.src.models.hrm_text.hrm_text_causal_lm_preprocessor import HrmTextCausalLMPreprocessor
from tools.checkpoint_conversion.convert_hrm_text_checkpoints import (
    convert_tokenizer,
    convert_weights,
    create_backbone,
    validate_tiny_gradient_parity,
)

MODEL_ID = "sapientinc/HRM-Text-1B"
REVISION = "9f082d68b8cd0ebc56e33f1c88c45609174c272c"
keras.config.set_dtype_policy("float32")
tiny_gradient_parity = validate_tiny_gradient_parity()
hf_tokenizer = AutoTokenizer.from_pretrained(MODEL_ID, revision=REVISION)
hf_model = AutoModelForCausalLM.from_pretrained(
    MODEL_ID, revision=REVISION, dtype=torch.float32
).cuda().eval()

backbone = create_backbone(hf_model.config)
conversion = convert_weights(backbone, hf_model)
tokenizer = convert_tokenizer(hf_tokenizer)
preprocessor = HrmTextCausalLMPreprocessor(tokenizer)
model = HrmTextCausalLM(backbone, preprocessor=None)
ACTIVE_INFERENCE_TOKENS = {
    "start": "<|im_start|>",
    "prefix_end": "<|im_end|>",
    "end": "<|box_end|>",
    "pad": "<|endoftext|>",
    "direct": "<|object_ref_start|>",
    "cot": "<|object_ref_end|>",
    "noisy": "<|quad_start|>",
    "synth": "<|quad_end|>",
}
for role, special_token in ACTIVE_INFERENCE_TOKENS.items():
    keras_ids = np.asarray(tokenizer([special_token])[0], dtype="int32").tolist()
    hf_ids = hf_tokenizer(special_token, add_special_tokens=False)["input_ids"]
    assert keras_ids == hf_ids == [tokenizer.token_to_id(special_token)], role
CONDITIONS = {name: ACTIVE_INFERENCE_TOKENS[name] for name in ("direct", "cot", "noisy", "synth")}
def make_prompt(instruction, conditions=("synth", "cot")):
    return f"<|im_start|>{''.join(CONDITIONS[name] for name in conditions)}{instruction}<|im_end|>"
for condition_name in CONDITIONS:
    prompt = make_prompt("Tokenization parity check.", (condition_name,))
    assert np.asarray(tokenizer([prompt])[0], dtype="int32").tolist() == hf_tokenizer(prompt, add_special_tokens=False)["input_ids"], condition_name
print(f"Keras backend: {keras.config.backend()}")
print(f"Logical KV-cache slots: {backbone.cache_slots}")
print(f"Mapped tensors: {conversion['mapped_tensors']}")
print(f"Exact parameter count: {conversion['keras_parameter_count']:,}")
print(f"Tiny gradient max absolute error: {tiny_gradient_parity['gradient_max_abs']:.3e}")


## Recurrent cache verification

The same 32 parameterized Transformer blocks are invoked recurrently through 128 logical cache slots. This check prefills a PrefixLM prompt and compares four cached response-token steps with complete forward passes over the growing sequence.

In [ ]:
condition = "<|quad_end|><|object_ref_end|>"  # synth,cot
prompt = f"<|im_start|>{condition}What is the capital of France?<|im_end|>"
prompt_ids = np.array([tokenizer([prompt])[0]], dtype="int32")
assert prompt_ids[0].tolist() == hf_tokenizer(prompt, add_special_tokens=False)["input_ids"]
response_text = " Paris is the capital of France."
response_ids = np.array([tokenizer([response_text])[0][:4]], dtype="int32")
assert response_ids[0].tolist() == hf_tokenizer(response_text, add_special_tokens=False)["input_ids"][:4]
cache_length = prompt_ids.shape[1] + response_ids.shape[1]
cache = keras.ops.zeros(
    (1, backbone.cache_slots, 2, cache_length, backbone.num_attention_heads, backbone.head_dim),
    dtype=model.compute_dtype,
)
_, _, cache = model.call_with_cache(
    prompt_ids, cache, 0, token_type_ids=np.ones_like(prompt_ids)
)
cache_errors = []
for step in range(response_ids.shape[1]):
    index = prompt_ids.shape[1] + step
    cached_logits, _, cache = model.call_with_cache(
        response_ids[:, step : step + 1], cache, index
    )
    full_ids = np.concatenate((prompt_ids, response_ids[:, : step + 1]), axis=1)
    full_types = np.concatenate(
        (np.ones_like(prompt_ids), np.zeros((1, step + 1), dtype="int32")), axis=1
    )
    full_logits = model({
        "token_ids": full_ids,
        "padding_mask": np.ones_like(full_ids),
        "token_type_ids": full_types,
    })[:, -1:]
    cache_errors.append(float(keras.ops.convert_to_numpy(
        keras.ops.max(keras.ops.abs(cached_logits - full_logits))
    )))
assert max(cache_errors) < 2e-4
print("Per-step cached/full maximum absolute errors:", cache_errors)


## Numerical verification

The conversion rejects any unmapped tensor. This check then compares all logits for a short input. The expected maximum absolute error is below `2e-4`; the initial port measured `1.29e-5` on a T4.

In [ ]:
text = "HRM-Text validates a portable Keras implementation."
hf_inputs = hf_tokenizer(text, return_tensors="pt").to("cuda")
hf_inputs["token_type_ids"] = torch.zeros_like(hf_inputs["input_ids"])
with torch.inference_mode():
    hf_logits = hf_model(**hf_inputs).logits.float().cpu().numpy()

keras_inputs = {
    "token_ids": hf_inputs["input_ids"].cpu().numpy(),
    "padding_mask": hf_inputs["attention_mask"].cpu().numpy(),
    "token_type_ids": hf_inputs["token_type_ids"].cpu().numpy(),
}
keras_logits = keras.ops.convert_to_numpy(model(keras_inputs))
max_abs_error = np.max(np.abs(keras_logits - hf_logits))
np.testing.assert_allclose(keras_logits, hf_logits, atol=2e-4, rtol=2e-4)
conversion["logit_max_abs"] = float(max_abs_error)
print(f"Forward parity passed; max absolute logit error: {max_abs_error:.3e}")
del hf_model
gc.collect()
torch.cuda.empty_cache()


## PrefixLM generation

The full prompt is a PrefixLM condition block (`token_type_ids == 1`). `generate()` uses the model's logical recurrent KV cache internally; the 1B configuration has `16 × 2 × (3 + 1) = 128` cache slots. Output quality is intentionally not evaluated here because this is a base, pre-alignment checkpoint.

In [ ]:
condition = "<|quad_end|><|object_ref_end|>"  # synth,cot
prompt = f"<|im_start|>{condition}What is the capital of France?<|im_end|>"
prompt_ids = np.asarray(tokenizer([prompt])[0], dtype="int32")
assert prompt_ids.tolist() == hf_tokenizer(prompt, add_special_tokens=False)["input_ids"]
# Four tokens are enough to exercise cached decoding on a T4 runtime.
max_length = len(prompt_ids) + 4
token_ids = np.full((1, max_length), tokenizer.pad_token_id, dtype="int32")
token_ids[0, : len(prompt_ids)] = prompt_ids
padding_mask = np.zeros((1, max_length), dtype="int32")
padding_mask[0, : len(prompt_ids)] = 1
prefix_mask = padding_mask.copy()

model.compile(sampler="greedy")
generated = model.generate(
    {
        "token_ids": token_ids,
        "padding_mask": padding_mask,
        "token_type_ids": prefix_mask,
    },
    max_length=max_length,
    stop_token_ids=None,
)
generated_text = tokenizer.detokenize(generated["token_ids"][0])
print(generated_text)


## Runtime measurements

These measurements characterize the portable float32 attention path on this runtime; they are not cross-hardware performance claims. Prefill reports complete forward-pass throughput. Cached decoding uses a 128-token PrefixLM prefill followed by 16 single-token cache updates.

In [ ]:
rng = np.random.default_rng(2026)
benchmark = {
    "device": torch.cuda.get_device_name(0),
    "dtype": model.compute_dtype,
}
def synchronize():
    torch.cuda.synchronize()

with torch.inference_mode():
    warmup = {
        "token_ids": rng.integers(16, 1000, (1, 8), dtype=np.int32),
        "padding_mask": np.ones((1, 8), dtype=np.int32),
        "token_type_ids": np.zeros((1, 8), dtype=np.int32),
    }
    model(warmup)
    synchronize()
    for length in (128, 512, 1024):
        inputs = {
            "token_ids": rng.integers(16, 1000, (1, length), dtype=np.int32),
            "padding_mask": np.ones((1, length), dtype=np.int32),
            "token_type_ids": np.zeros((1, length), dtype=np.int32),
        }
        torch.cuda.reset_peak_memory_stats()
        synchronize()
        started = time.perf_counter()
        model(inputs)
        synchronize()
        elapsed = time.perf_counter() - started
        benchmark[f"prefill_{length}"] = {
            "seconds": elapsed,
            "tokens_per_second": length / elapsed,
            "peak_torch_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
        }

    prefill_length, decode_steps = 128, 16
    benchmark_cache_length = prefill_length + decode_steps
    benchmark_cache = keras.ops.zeros(
        (1, backbone.cache_slots, 2, benchmark_cache_length, backbone.num_attention_heads, backbone.head_dim),
        dtype=model.compute_dtype,
    )
    benchmark_prompt = rng.integers(16, 1000, (1, prefill_length), dtype=np.int32)
    torch.cuda.reset_peak_memory_stats()
    synchronize()
    started = time.perf_counter()
    _, benchmark_cache = backbone.call_with_cache(
        benchmark_prompt, benchmark_cache, 0, token_type_ids=np.ones_like(benchmark_prompt)
    )
    synchronize()
    cache_prefill_seconds = time.perf_counter() - started
    step_times = []
    for index in range(prefill_length, benchmark_cache_length):
        token = rng.integers(16, 1000, (1, 1), dtype=np.int32)
        synchronize()
        started = time.perf_counter()
        _, benchmark_cache = backbone.call_with_cache(token, benchmark_cache, index)
        synchronize()
        step_times.append(time.perf_counter() - started)
    benchmark["cached_decode"] = {
        "prefill_seconds": cache_prefill_seconds,
        "decode_steps": decode_steps,
        "seconds_per_token": sum(step_times) / len(step_times),
        "tokens_per_second": len(step_times) / sum(step_times),
        "cache_gib": int(np.prod(benchmark_cache.shape)) * 4 / 2**30,
        "peak_torch_allocated_gib": torch.cuda.max_memory_allocated() / 2**30,
    }
print(json.dumps(benchmark, indent=2))


## Save and reload the converted preset

The converter must produce a complete local preset, not only in-memory weights. This final check saves the backbone, task, preprocessor, and tokenizer; releases the original model; reloads the preset through the public API; and verifies unchanged logits and special-token IDs.

In [ ]:
preset_dir = "/content/hrm_text_1b"
model.preprocessor = preprocessor
model.save_to_preset(preset_dir)
expected_special_token_ids = {
    "start": tokenizer.start_token_id,
    "prefix_end": tokenizer.prefix_end_token_id,
    "end": tokenizer.end_token_id,
    "pad": tokenizer.pad_token_id,
    "direct": tokenizer.direct_condition_token_id,
    "cot": tokenizer.cot_condition_token_id,
    "noisy": tokenizer.noisy_condition_token_id,
    "synth": tokenizer.synth_condition_token_id,
}
del model, backbone, cache, benchmark_cache
gc.collect()
keras.backend.clear_session()
torch.cuda.empty_cache()
restored = HrmTextCausalLM.from_preset(preset_dir)
restored_logits = keras.ops.convert_to_numpy(restored(keras_inputs))
restored_max_abs_error = float(np.max(np.abs(restored_logits - hf_logits)))
np.testing.assert_allclose(restored_logits, hf_logits, atol=2e-4, rtol=2e-4)
restored_tokenizer = restored.preprocessor.tokenizer
actual_special_token_ids = {
    "start": restored_tokenizer.start_token_id,
    "prefix_end": restored_tokenizer.prefix_end_token_id,
    "end": restored_tokenizer.end_token_id,
    "pad": restored_tokenizer.pad_token_id,
    "direct": restored_tokenizer.direct_condition_token_id,
    "cot": restored_tokenizer.cot_condition_token_id,
    "noisy": restored_tokenizer.noisy_condition_token_id,
    "synth": restored_tokenizer.synth_condition_token_id,
}
assert actual_special_token_ids == expected_special_token_ids
results = {
    "source_revision": KERAS_HUB_REVISION,
    "checkpoint_revision": REVISION,
    "conversion": conversion,
    "tiny_gradient_parity": tiny_gradient_parity,
    "cache_slots": 128,
    "cache_step_max_abs_errors": cache_errors,
    "restored_max_abs_logit_error": restored_max_abs_error,
    "special_token_ids": actual_special_token_ids,
    "benchmark": benchmark,
    "generated_text": str(generated_text),
}
with open("/content/hrm_text_1b_results.json", "w") as file:
    json.dump(results, file, indent=2)
print(f"Preset round-trip passed; max absolute logit error: {restored_max_abs_error:.3e}")
